# Theme 2: Bug-Fix Quality Over Time

**RQ2:** How does bug-fix *acceptance rate* change over time?  
**RQ3:** How does *time to merge* change over time?  
**RQ4:** How does *patch size* change over time?  
**RQ5:** How does *revision burden* change over time?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)

**Methodology notes:**
- **Confidence bands:** monthly rates carry Wilson 95% CIs; monthly medians carry bootstrap 95% CIs. Bands are shaded around each line.
- **Minimum support:** a monthly cell is reported only if it has ≥ `MIN_N_PROP` (30) PRs for a rate, or ≥ `MIN_N_MEDIAN` (20) for a median. Sparse agent-months (especially Devin) correctly drop out instead of producing 1–2-PR spikes.
- **Repo-matched baseline:** RQ2–RQ4 are re-computed on a repo- and time-matched human set (`build_matched_human_baseline`) so agent-vs-human gaps are not driven by repo mix.
- **Survivorship cutoff** (last 30 days dropped) is applied by `load_fix_prs`.

In [1]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Mahmoudabujadallah\CLI\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    odds_ratio_ci, cliffs_delta, bh_correct,
    wilson_ci, bootstrap_median_ci, classify_file_role,
    build_matched_human_baseline, annotate_model_releases,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
    MIN_N_PROP, MIN_N_MEDIAN,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [3]:
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs()
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()

# Repo- and time-matched human baseline (for the matched-vs-unmatched comparison)
matched    = build_matched_human_baseline(df)
m_human_df = matched[~matched['is_agent']].copy()

# Commit data needed for RQ4 / RQ5
commits   = load_commits()
details   = load_commit_details()
rev_stats = build_revision_stats(df, commits, details)
print('All data loaded.')

Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos
  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs
  Fix PRs loaded: 371,577  |  Agent: 108,080  |  Human: 263,497


  Matched baseline: 108,080 agent PRs + 52,418 matched human PRs (from 263,497 human PRs)
Loading commits from HuggingFace ...


  Commits loaded: 1,156,238
Loading commit details from HuggingFace ...


  Commit details loaded: 7,451,150


All data loaded.


## RQ2: How does bug-fix acceptance rate change over time?

In [4]:
# Monthly merge rate per agent + human baseline, with Wilson 95% CIs.
# A cell is reported only if it has >= MIN_N_PROP PRs.
months = sorted(df['month'].unique())
idx    = [str(m) for m in months]

def _month_grp(m, g, agents_src=agents_df, human_src=human_df):
    if g == 'Human':
        return human_src[human_src['month'] == m]
    return agents_src[(agents_src['month'] == m) & (agents_src['agent'] == g)]

# point estimate + CI bounds per group, kept in parallel dicts for the figure
rate, rate_lo, rate_hi = {}, {}, {}
for g in AGENTS + ['Human']:
    rate[g], rate_lo[g], rate_hi[g] = [], [], []
    for m in months:
        sub = _month_grp(m, g)
        n, k = len(sub), int(sub['is_merged'].sum())
        if n >= MIN_N_PROP:
            lo, hi = wilson_ci(k, n)
            rate[g].append(k / n * 100); rate_lo[g].append(lo); rate_hi[g].append(hi)
        else:
            rate[g].append(None); rate_lo[g].append(None); rate_hi[g].append(None)

monthly_rate = pd.DataFrame(rate, index=idx)
print(f'Monthly merge rate (%), reported only where n >= {MIN_N_PROP}:')
print(monthly_rate.round(1).to_string())

Monthly merge rate (%), reported only where n >= 30:
         Copilot  Cursor  Claude_Code  Devin  Human
2024-12     91.3    94.5         83.4   58.0   87.9
2025-01     91.7    96.2         83.5   59.2   87.3
2025-02     95.8    93.6         83.4   59.6   88.0
2025-03     95.3    94.2         78.4   46.5   87.0
2025-04     93.1    94.1         84.5   70.6   88.0
2025-05     73.5    88.8         84.6   75.8   87.0
2025-06     78.3    88.7         87.6   70.5   85.8
2025-07     78.3    89.3         88.1   69.5   84.4
2025-08     79.7    86.9         88.8   77.7   83.9
2025-09     73.5    88.4         82.7   65.8   84.0
2025-10     76.7    89.0         86.5   65.6   83.5
2025-11     77.7    87.3         85.3   52.7   83.9
2025-12     76.3    89.6         89.7   74.3   83.7
2026-01     75.5    88.8         89.6   77.4   82.8


In [5]:
# Figure: monthly acceptance rate per agent vs human, with Wilson 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(rate[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(rate_lo[agent], dtype=float),
                    np.array(rate_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.plot(idx, np.array(rate['Human'], dtype=float), 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.fill_between(x, np.array(rate_lo['Human'], dtype=float),
                np.array(rate_hi['Human'], dtype=float),
                color=AGENT_COLORS['Human'], alpha=0.12)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ2: Monthly Bug-Fix Acceptance Rate per Agent vs Human (Wilson 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq2_monthly_merge_rate', THEME2_DIR)

  -> Saved: results\theme2_figures\rq2_monthly_merge_rate.png


WindowsPath('results/theme2_figures/rq2_monthly_merge_rate.png')

## RQ3: How does time to merge change over time?

In [6]:
# Monthly median time-to-merge per agent + human, with bootstrap 95% CIs.
merged_agents = agents_df[agents_df['is_merged']]
merged_human  = human_df[human_df['is_merged']]

def _month_grp_merged(m, g):
    if g == 'Human':
        return merged_human[merged_human['month'] == m]
    return merged_agents[(merged_agents['month'] == m) & (merged_agents['agent'] == g)]

ttm, ttm_lo, ttm_hi = {}, {}, {}
for g in AGENTS + ['Human']:
    ttm[g], ttm_lo[g], ttm_hi[g] = [], [], []
    for m in months:
        sub = _month_grp_merged(m, g)['hours_to_merge']
        if len(sub) >= MIN_N_MEDIAN:
            lo, hi = bootstrap_median_ci(sub)
            ttm[g].append(round(sub.median(), 2)); ttm_lo[g].append(lo); ttm_hi[g].append(hi)
        else:
            ttm[g].append(None); ttm_lo[g].append(None); ttm_hi[g].append(None)

monthly_ttm = pd.DataFrame(ttm, index=idx)
print(f'Monthly median time to merge (hours), reported only where n >= {MIN_N_MEDIAN}:')
print(monthly_ttm.to_string())

Monthly median time to merge (hours), reported only where n >= 20:
         Copilot  Cursor  Claude_Code  Devin  Human
2024-12     2.05    0.56         2.32   0.46   5.27
2025-01     1.98    0.52         2.92   0.31   5.28
2025-02     0.75    0.75         1.03   0.07   4.45
2025-03     4.82    0.52         1.35   0.23   5.56
2025-04     1.81    0.73         1.26   0.20   4.98
2025-05     0.51    0.41         0.80   0.22   5.41
2025-06     0.59    0.15         0.26   0.21   5.10
2025-07     0.52    0.04         0.27   0.23   5.49
2025-08     0.85    0.24         0.64   0.40   6.09
2025-09     1.37    0.53         1.10   1.98   5.70
2025-10     2.32    0.59         1.09   3.10   6.59
2025-11     1.95    0.46         1.20   2.67   6.23
2025-12     1.82    0.54         0.90   1.33   5.34
2026-01     2.48    0.54         0.99   2.42   5.37


In [7]:
# Figure: monthly time to merge, with bootstrap 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(ttm[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(ttm_lo[agent], dtype=float),
                    np.array(ttm_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.plot(idx, np.array(ttm['Human'], dtype=float), 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.fill_between(x, np.array(ttm_lo['Human'], dtype=float),
                np.array(ttm_hi['Human'], dtype=float),
                color=AGENT_COLORS['Human'], alpha=0.12)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Time to Merge (hours)')
ax.set_title('RQ3: Monthly Median Time to Merge per Agent vs Human (bootstrap 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq3_monthly_time_to_merge', THEME2_DIR)

  -> Saved: results\theme2_figures\rq3_monthly_time_to_merge.png


WindowsPath('results/theme2_figures/rq3_monthly_time_to_merge.png')

## RQ4: How does bug-fix patch size change over time?

In [8]:
# Patch size split by file role. A single "lines added" number conflates production
# code, tests, and generated/vendored files; agents and humans write different mixes,
# so we classify each changed file and aggregate lines per role per PR.
details['role'] = details['filename'].map(classify_file_role)

role_size = (
    details.groupby(['pr_id', 'role'])['additions'].sum()
    .unstack(fill_value=0)
    .rename(columns=lambda r: f'{r}_added')
    .reset_index()
    .rename(columns={'pr_id': 'id'})
)
for col in ['prod_added', 'test_added', 'generated_added']:
    if col not in role_size:
        role_size[col] = 0.0
role_size['total_added'] = role_size[['prod_added', 'test_added', 'generated_added']].sum(axis=1)

df_size     = df.merge(role_size, on='id', how='left')
agents_size = df_size[df_size['is_agent'] & df_size['agent'].isin(AGENTS)]
human_size  = df_size[~df_size['is_agent']]

# Monthly median PRODUCTION lines added (the cleaned patch-size signal), bootstrap CIs
psize, psize_lo, psize_hi = {}, {}, {}
for g in AGENTS + ['Human']:
    psize[g], psize_lo[g], psize_hi[g] = [], [], []
    for m in months:
        if g == 'Human':
            sub = human_size[human_size['month'] == m]['prod_added']
        else:
            sub = agents_size[(agents_size['month'] == m) & (agents_size['agent'] == g)]['prod_added']
        if len(sub) >= MIN_N_MEDIAN:
            lo, hi = bootstrap_median_ci(sub)
            psize[g].append(round(sub.median(), 1)); psize_lo[g].append(lo); psize_hi[g].append(hi)
        else:
            psize[g].append(None); psize_lo[g].append(None); psize_hi[g].append(None)

monthly_size = pd.DataFrame(psize, index=idx)
print(f'Monthly median PRODUCTION lines added, reported only where n >= {MIN_N_MEDIAN}:')
print(monthly_size.to_string())

Monthly median PRODUCTION lines added, reported only where n >= 20:
         Copilot  Cursor  Claude_Code  Devin  Human
2024-12     12.0    14.0          2.0   51.5   10.0
2025-01     12.0    10.0          2.0   38.5   11.0
2025-02     13.0    12.0          2.0   24.0   11.0
2025-03      8.5    12.5          1.0   17.0   11.0
2025-04     17.0    13.0          3.0   17.0   11.0
2025-05     21.0    13.0          5.0   25.0   11.0
2025-06     23.0    16.0         11.0   22.0   12.0
2025-07     38.0    47.0         20.0   21.0   12.0
2025-08     49.0    28.0          9.0   22.0   12.0
2025-09     34.0    22.0          4.0   30.0   13.0
2025-10     35.0    20.0          6.0   15.0   12.0
2025-11     24.0    20.0          4.0   28.0   14.0
2025-12     24.0    20.0          6.0   15.0   14.0
2026-01     24.0    22.0          6.0   10.0   14.0


In [9]:
# Figure: monthly PRODUCTION patch size (lines added), bootstrap 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(psize[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(psize_lo[agent], dtype=float),
                    np.array(psize_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.plot(idx, np.array(psize['Human'], dtype=float), 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.fill_between(x, np.array(psize_lo['Human'], dtype=float),
                np.array(psize_hi['Human'], dtype=float),
                color=AGENT_COLORS['Human'], alpha=0.12)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Production Lines Added')
ax.set_title('RQ4: Monthly Median Patch Size — Production Code Only (bootstrap 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq4_monthly_patch_size', THEME2_DIR)

  -> Saved: results\theme2_figures\rq4_monthly_patch_size.png


WindowsPath('results/theme2_figures/rq4_monthly_patch_size.png')

In [10]:
# Figure: overall median lines added by file role (prod / test / generated) per group.
# Shows how much of each group's patch is production vs test vs generated code.
roles  = ['prod_added', 'test_added', 'generated_added']
labels = ['Production', 'Test', 'Generated']
groups = AGENTS + ['Human']

medians = {role: [] for role in roles}
for g in groups:
    sub = human_size if g == 'Human' else agents_size[agents_size['agent'] == g]
    for role in roles:
        medians[role].append(sub[role].median())

xpos = np.arange(len(groups))
width = 0.25
fig, ax = plt.subplots(figsize=(11, 5))
for i, (role, lab) in enumerate(zip(roles, labels)):
    ax.bar(xpos + (i - 1) * width, medians[role], width, label=lab)
ax.set_xticks(xpos)
ax.set_xticklabels(groups, rotation=20, ha='right')
ax.set_ylabel('Median Lines Added')
ax.set_title('RQ4: Median Patch Size by File Role per Group')
ax.legend()
fig.tight_layout()
save_fig(fig, 'rq4_patch_size_by_role', THEME2_DIR)

  -> Saved: results\theme2_figures\rq4_patch_size_by_role.png


WindowsPath('results/theme2_figures/rq4_patch_size_by_role.png')

## RQ5: How does revision burden change over time?

In [11]:
# Monthly revision rate: % of merged PRs with >1 commit, per agent, with Wilson 95% CIs.
# NOTE: revision burden is measured from commit count + revision lines only; the dataset
# has no review-comment counts, so review round-trips are not observable (see threats).
agent_rev = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
agent_rev['is_revised'] = agent_rev['num_commits'] > 1

rev, rev_lo, rev_hi = {}, {}, {}
for g in AGENTS:
    rev[g], rev_lo[g], rev_hi[g] = [], [], []
    for m in months:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == g)]
        n, k = len(sub), int(sub['is_revised'].sum())
        if n >= MIN_N_PROP:
            lo, hi = wilson_ci(k, n)
            rev[g].append(round(k / n * 100, 1)); rev_lo[g].append(lo); rev_hi[g].append(hi)
        else:
            rev[g].append(None); rev_lo[g].append(None); rev_hi[g].append(None)

monthly_rev = pd.DataFrame(rev, index=idx)
print(f'Monthly revision rate (%), reported only where n >= {MIN_N_PROP}:')
print(monthly_rev.to_string())

Monthly revision rate (%), reported only where n >= 30:
         Copilot  Cursor  Claude_Code  Devin
2024-12     32.2    44.5         26.5   55.1
2025-01     37.7    42.7         32.7   56.8
2025-02     35.8    43.3         28.7   43.1
2025-03     32.2    46.4         24.6   40.6
2025-04     50.0    49.9         34.7   38.7
2025-05     91.0    41.3         33.9   41.7
2025-06     91.6    36.6         36.7   39.6
2025-07     91.6    35.8         43.7   39.7
2025-08     96.3    43.0         39.4   46.1
2025-09     93.9    46.8         39.9   62.5
2025-10     94.1    46.8         38.6   61.0
2025-11     92.9    46.8         37.3   65.8
2025-12     90.7    47.2         34.3   56.1
2026-01     90.0    47.4         35.1   47.9


In [12]:
# Figure: monthly revision rate, with Wilson 95% CI bands
x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(rev[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(rev_lo[agent], dtype=float),
                    np.array(rev_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Revision Rate (%)')
ax.set_title('RQ5: Monthly Revision Rate per Agent (Wilson 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_rate', THEME2_DIR)

  -> Saved: results\theme2_figures\rq5_monthly_revision_rate.png


WindowsPath('results/theme2_figures/rq5_monthly_revision_rate.png')

In [13]:
# Monthly median revision lines added (revised PRs only), with bootstrap 95% CIs
reff, reff_lo, reff_hi = {}, {}, {}
for g in AGENTS:
    reff[g], reff_lo[g], reff_hi[g] = [], [], []
    for m in months:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == g)
                        & (agent_rev['num_commits'] > 1)]['rev_lines_added']
        if len(sub) >= MIN_N_MEDIAN:
            lo, hi = bootstrap_median_ci(sub)
            reff[g].append(round(sub.median(), 1)); reff_lo[g].append(lo); reff_hi[g].append(hi)
        else:
            reff[g].append(None); reff_lo[g].append(None); reff_hi[g].append(None)

monthly_revsize = pd.DataFrame(reff, index=idx)

x = list(range(len(idx)))
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(idx, np.array(reff[agent], dtype=float), 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
    ax.fill_between(x, np.array(reff_lo[agent], dtype=float),
                    np.array(reff_hi[agent], dtype=float),
                    color=AGENT_COLORS[agent], alpha=0.15)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Revision Lines Added')
ax.set_title('RQ5: Monthly Median Revision Effort — Revised PRs (bootstrap 95% CI)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_effort', THEME2_DIR)

  -> Saved: results\theme2_figures\rq5_monthly_revision_effort.png


WindowsPath('results/theme2_figures/rq5_monthly_revision_effort.png')

## Matched-baseline sensitivity (RQ2)

Agents cluster in particular repos, so an agent-vs-human merge-rate gap can partly reflect *which repos* the comparison spans. Below, each agent's merge rate is tested against both the **unmatched** human baseline (all human PRs) and the **repo+time-matched** baseline. A large shift in the odds ratio between the two columns indicates the unmatched comparison was confounded by repo mix. Odds ratios carry 95% CIs; p-values are Benjamini-Hochberg adjusted across the four agents.

In [14]:
# Agent merge rate vs unmatched and repo+time-matched human baselines.
hu_m, hu_t, hu_r = merge_rate(human_df)     # unmatched humans
hm_m, hm_t, hm_r = merge_rate(m_human_df)   # matched humans

pvals_unm, pvals_mat, rows = [], [], []
for agent in AGENTS:
    sub = agents_df[agents_df['agent'] == agent]
    a_m, a_t, a_r = merge_rate(sub)
    or_u, lo_u, hi_u = odds_ratio_ci(a_m, a_t, hu_m, hu_t)
    or_m, lo_m, hi_m = odds_ratio_ci(a_m, a_t, hm_m, hm_t)
    _, p_u = chi_square(a_m, a_t, hu_m, hu_t); pvals_unm.append(p_u)
    _, p_m = chi_square(a_m, a_t, hm_m, hm_t); pvals_mat.append(p_m)
    rows.append((agent, a_r, or_u, lo_u, hi_u, or_m, lo_m, hi_m))

padj_u = bh_correct(pvals_unm)
padj_m = bh_correct(pvals_mat)

print(f"{'Agent':<13}{'Rate%':>7}   {'OR vs unmatched human':<28}{'OR vs matched human':<28}")
print('-' * 78)
for (agent, a_r, or_u, lo_u, hi_u, or_m, lo_m, hi_m), pu, pm in zip(rows, padj_u, padj_m):
    s_u = f"{or_u:.2f} [{lo_u:.2f},{hi_u:.2f}] {sig_label(pu)}"
    s_m = f"{or_m:.2f} [{lo_m:.2f},{hi_m:.2f}] {sig_label(pm)}"
    print(f"{agent:<13}{a_r:>6.1f}%   {s_u:<28}{s_m:<28}")
print(f"\nHuman merge rate — unmatched: {hu_r:.1f}% (n={hu_t:,})  |  matched: {hm_r:.1f}% (n={hm_t:,})")
print("BH-adjusted p; OR>1 => agent merges more often than that human baseline.")

Agent          Rate%   OR vs unmatched human       OR vs matched human         
------------------------------------------------------------------------------
Copilot        77.7%   0.60 [0.58,0.62] ***        0.55 [0.53,0.57] ***        
Cursor         89.8%   1.51 [1.45,1.56] ***        1.40 [1.34,1.46] ***        
Claude_Code    86.2%   1.07 [1.04,1.10] ***        0.99 [0.95,1.03] ns         
Devin          66.4%   0.34 [0.32,0.36] ***        0.31 [0.30,0.33] ***        

Human merge rate — unmatched: 85.4% (n=263,497)  |  matched: 86.3% (n=52,418)
BH-adjusted p; OR>1 => agent merges more often than that human baseline.


## Threats to validity (Theme 2)

- **Merge rate ≠ quality.** A merged PR is not necessarily correct, and a closed-unmerged PR is not necessarily wrong (it may be a duplicate or superseded). RQ2 measures acceptance, not correctness.
- **Auto-merge is unobservable.** The dataset has no `auto_merge` flag, no `merged_by`, and the commits table has no timestamp, so we cannot separate human-reviewed merges from bot/auto-merges. RQ2/RQ3 therefore partly reflect each repo's *merge policy* rather than agent skill — a confound we cannot net out here.
- **Time-to-merge reflects workflow, not just effort.** Fast merges can mean trivial diffs, CI-gated auto-merge, or an attentive maintainer; slow merges can mean a busy maintainer, not a hard fix.
- **Revision burden is commit-based.** `num_commits > 1` is sensitive to force-push and squash-merge policies, and review round-trips (review-comment counts) are absent from the data. Treat RQ5 as a coarse proxy.
- **First-commit ordering is fragile.** `build_revision_stats` takes the "first commit per PR" by parquet row order because the commits table has no commit date; if storage order is not chronological, revision-line attribution can be off.
- **Patch size still imperfect after role split.** `classify_file_role` is heuristic (path patterns); misclassified files leak between prod/test/generated buckets.
- **Repo mix.** Compare the unmatched vs matched odds ratios above; where they diverge, the unmatched agent-vs-human number was partly a repo-composition artefact.